# Busan Traffic AI PBL - 17_busan_hyperparameter_tuning.ipynb

부산시 교통량 예측 프로젝트 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
# =========================
# [셀 1] 라이브러리 로드
# =========================
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# =========================
# [셀 2] 데이터 로드/병합/split
# =========================
X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

y_candidates = [c for c in df_y.columns if c != "date"]
y_col = y_candidates[0] if len(y_candidates) == 1 else ( "y" if "y" in y_candidates else y_candidates[-1] )

df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)

train_mask = (df["date"] >= "2022-01-01") & (df["date"] <= "2022-12-31")
val_mask   = (df["date"] >= "2023-01-01") & (df["date"] <= "2023-01-31")
test_mask  = (df["date"] >= "2023-02-01") & (df["date"] <= "2023-12-31")

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

feature_cols = [c for c in df.columns if c not in ["date", y_col]]
X_train, y_train = df_train[feature_cols], df_train[y_col]
X_val, y_val     = df_val[feature_cols], df_val[y_col]
X_test, y_test   = df_test[feature_cols], df_test[y_col]

In [ ]:
# =========================
# [셀 3] 전처리
# =========================
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent"))
        ]), cat_cols),
    ],
    remainder="drop"
)

In [ ]:
# =========================
# [셀 4] Baseline vs Tuned 비교를 위한 공통 함수
# =========================
def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

In [ ]:
# =========================
# [셀 5] 튜닝 1: kNN
# =========================
pipe_knn = Pipeline([("preprocess", preprocess), ("model", KNeighborsRegressor())])

param_grid_knn = {
    "model__n_neighbors": [3,5,7,9,15],
    "model__weights": ["uniform", "distance"]
}

gs_knn = GridSearchCV(
    pipe_knn,
    param_grid=param_grid_knn,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
gs_knn.fit(X_train, y_train)

print("best params:", gs_knn.best_params_)
val_pred = gs_knn.predict(X_val)
test_pred = gs_knn.predict(X_test)
print("VAL", metrics(y_val, val_pred))
print("TEST", metrics(y_test, test_pred))

In [ ]:
# =========================
# [셀 6] 튜닝 2: Ridge
# =========================
pipe_ridge = Pipeline([("preprocess", preprocess), ("model", Ridge(random_state=42))])

param_grid_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

gs_ridge = GridSearchCV(
    pipe_ridge,
    param_grid=param_grid_ridge,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
gs_ridge.fit(X_train, y_train)

print("best params:", gs_ridge.best_params_)
val_pred = gs_ridge.predict(X_val)
test_pred = gs_ridge.predict(X_test)
print("VAL", metrics(y_val, val_pred))
print("TEST", metrics(y_test, test_pred))

In [ ]:
# =========================
# [셀 7] 튜닝 3: RandomForest
# =========================
pipe_rf = Pipeline([("preprocess", preprocess), ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])

param_grid_rf = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_split": [2, 5],
}

gs_rf = GridSearchCV(
    pipe_rf,
    param_grid=param_grid_rf,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
gs_rf.fit(X_train, y_train)

print("best params:", gs_rf.best_params_)
val_pred = gs_rf.predict(X_val)
test_pred = gs_rf.predict(X_test)
print("VAL", metrics(y_val, val_pred))
print("TEST", metrics(y_test, test_pred))

In [ ]:
# =========================
# [셀 7] 튜닝 3: RandomForest
# =========================
pipe_rf = Pipeline([("preprocess", preprocess), ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])

param_grid_rf = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_split": [2, 3, 5],
}

gs_rf = GridSearchCV(
    pipe_rf,
    param_grid=param_grid_rf,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
gs_rf.fit(X_train, y_train)

print("best params:", gs_rf.best_params_)
val_pred = gs_rf.predict(X_val)
test_pred = gs_rf.predict(X_test)
print("VAL", metrics(y_val, val_pred))
print("TEST", metrics(y_test, test_pred))